# Adım 6: Makine Öğrenmesi - Çoklu Model Karşılaştırma

Bu notebook, Spotify feature tablosu üzerinden **5 regresyon modelini** eğitir, karşılaştırır ve tüm sonuçları **MLflow** ile takip eder.

Modeller:
1. Linear Regression
2. Decision Tree Regressor
3. Random Forest Regressor
4. GBT Regressor
5. Generalized Linear Regression (GLR)

Hedef değişken: `popularity`


## 1. Kütüphaneler ve Spark Oturumu


In [7]:
import os
import tempfile

import mlflow
import mlflow.spark
import pandas as pd
import numpy as np
import matplotlib
%matplotlib inline
import matplotlib.pyplot as plt

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, mean, stddev_samp, abs as spark_abs

from pyspark.ml.regression import (
    LinearRegression,
    DecisionTreeRegressor,
    RandomForestRegressor,
    GBTRegressor,
    GeneralizedLinearRegression
)
from pyspark.ml.evaluation import RegressionEvaluator

plt.rcParams.update({
    "figure.facecolor": "#1e1e2f",
    "axes.facecolor": "#1e1e2f",
    "axes.edgecolor": "#444466",
    "axes.labelcolor": "#d0d0e0",
    "text.color": "#d0d0e0",
    "xtick.color": "#a0a0c0",
    "ytick.color": "#a0a0c0",
    "grid.color": "#333355",
    "figure.dpi": 120,
    "font.size": 11,
    "axes.titlesize": 14,
    "axes.labelsize": 12,
})
PALETTE = ["#7c4dff", "#00e5ff", "#ff4081", "#ffab40", "#69f0ae"]

spark = (
    SparkSession.builder
    .appName("Spotify Step6 ML Model Comparison")
    .config("spark.jars.packages", "io.delta:delta-spark_2.12:3.2.0")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.sql.warehouse.dir", "/home/jovyan/work/spark-warehouse")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

print("Spark oturumu başlatıldı.")


Spark oturumu başlatıldı.


## 2. MLflow Konfigürasyonu


In [8]:
from datetime import datetime

MLFLOW_URI = os.environ.get("MLFLOW_TRACKING_URI", "http://mlflow:5001")

# Eski experiment artifact path'i local /mlflow olabilir. Varsayılanı her çalıştırmada yeni isim yapıyoruz.
default_exp_name = f"Spotify_Popularity_Regression_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
EXPERIMENT_NAME = os.environ.get("MLFLOW_EXPERIMENT_NAME", default_exp_name)

mlflow.set_tracking_uri(MLFLOW_URI)

# Eski açık run kaldıysa kapat
if mlflow.active_run() is not None:
    mlflow.end_run(status="KILLED")

mlflow.set_experiment(EXPERIMENT_NAME)
exp = mlflow.get_experiment_by_name(EXPERIMENT_NAME)

print(f"MLflow Tracking URI : {MLFLOW_URI}")
print(f"Experiment          : {EXPERIMENT_NAME}")
print(f"Experiment ID       : {exp.experiment_id}")
print(f"Artifact location   : {exp.artifact_location}")

if exp.artifact_location.startswith("/mlflow"):
    raise RuntimeError(
        "Bu experiment local '/mlflow' artifact path kullanıyor. "
        "Yeni bir experiment adıyla tekrar çalıştır (örn: MLFLOW_EXPERIMENT_NAME=Spotify_Popularity_Regression_v4)."
    )


2026/05/09 21:20:39 INFO mlflow.tracking.fluent: Experiment with name 'Spotify_Popularity_Regression_20260509_212039' does not exist. Creating a new experiment.


MLflow Tracking URI : http://mlflow:5000
Experiment          : Spotify_Popularity_Regression_20260509_212039
Experiment ID       : 2
Artifact location   : mlflow-artifacts:/2


## 3. Feature Tablosunu Okuma

Notebook iki olası path'i kontrol eder ve mevcut olanı seçer:
- `/home/jovyan/work/delta/gold/spotify_tracks_features`
- `/home/jovyan/work/delta/gold/spotify_tracks_features_ml`


In [9]:
PLOT_DIR = "/home/jovyan/work/notebooks/ml_plots"
REPORT_DIR = "/home/jovyan/work/reports"
os.makedirs(PLOT_DIR, exist_ok=True)
os.makedirs(REPORT_DIR, exist_ok=True)

candidate_paths = [
    "/home/jovyan/work/delta/gold/spotify_tracks_features",
    "/home/jovyan/work/delta/gold/spotify_tracks_features_ml",
]

FEATURES_PATH = None
for p in candidate_paths:
    if os.path.exists(p):
        FEATURES_PATH = p
        break

if FEATURES_PATH is None:
    raise FileNotFoundError(
        "Feature tablosu bulunamadı. Önce 5_Feature_Engineering.ipynb ile gold feature tablosunu üret."
    )

print(f"Kullanılan feature path: {FEATURES_PATH}")

df = (
    spark.read.format("delta").load(FEATURES_PATH)
    .filter(col("popularity").isNotNull())
)

print(f"Toplam kayıt sayısı : {df.count()}")
print(f"Kolon sayısı        : {len(df.columns)}")
df.select("track_id", "popularity", "features").show(5, truncate=False)


Kullanılan feature path: /home/jovyan/work/delta/gold/spotify_tracks_features
Toplam kayıt sayısı : 12447
Kolon sayısı        : 20
+----------------------+----------+----------------------------------------------------------------------------------------------------------------------------------+
|track_id              |popularity|features                                                                                                                          |
+----------------------+----------+----------------------------------------------------------------------------------------------------------------------------------+
|0QzY6rP7C8IWMnpEyuQmC6|56        |[193027.0,0.659,0.338,-10.367,0.0376,0.842,0.0,0.277,0.495,145.877,0.0,0.0,1.0,0.0,0.22274200000000002,0.16731000000000001,0.0]   |
|203ADb8800e1YSzyrFQUZ9|23        |[240449.0,0.524,0.566,-5.443,0.0364,0.0411,0.0,0.25,0.362,159.914,0.0,0.0,1.0,0.0,0.29658399999999996,0.20489199999999996,0.0]    |
|6b4mWoXrscyajbLOB2qwfG|29        

## 4. Train / Test Split


In [10]:
train_df, test_df = df.randomSplit([0.8, 0.2], seed=42)

train_count = train_df.count()
test_count = test_df.count()

print(f"Train kayıt sayısı : {train_count}")
print(f"Test kayıt sayısı  : {test_count}")


Train kayıt sayısı : 10037
Test kayıt sayısı  : 2410


## 5. Yardımcı Fonksiyonlar

Bu bölümde metrik hesaplama, residual analizi, feature importance ve MLflow log fonksiyonları hazırlanır.


In [11]:
# Vector içindeki feature sırası 5_Feature_Engineering notebook'u ile uyumlu
expected_feature_cols = [
    "duration_ms", "danceability", "energy", "loudness", "speechiness",
    "acousticness", "instrumentalness", "liveness", "valence", "tempo",
    "explicit_int", "track_genre_index", "artist_count", "is_short_track",
    "energy_dance_interaction", "mood_index", "is_feat"
]

vec_size = len(train_df.select("features").head()["features"])
if len(expected_feature_cols) == vec_size:
    feature_cols = expected_feature_cols
else:
    feature_cols = [f"feature_{i}" for i in range(vec_size)]

rmse_eval = RegressionEvaluator(labelCol="popularity", predictionCol="prediction", metricName="rmse")
mae_eval = RegressionEvaluator(labelCol="popularity", predictionCol="prediction", metricName="mae")
r2_eval = RegressionEvaluator(labelCol="popularity", predictionCol="prediction", metricName="r2")

results = []
model_importances = {}


def evaluate_metrics(pred_df):
    return {
        "rmse": float(rmse_eval.evaluate(pred_df)),
        "mae": float(mae_eval.evaluate(pred_df)),
        "r2": float(r2_eval.evaluate(pred_df)),
    }


def residual_stats_and_plot(pred_df, model_name):
    residual_df = (
        pred_df
        .select("popularity", "prediction")
        .withColumn("residual", col("popularity") - col("prediction"))
        .withColumn("abs_residual", spark_abs(col("residual")))
    )

    stats_row = residual_df.agg(
        mean("residual").alias("residual_mean"),
        stddev_samp("residual").alias("residual_std"),
        mean("abs_residual").alias("residual_abs_mean")
    ).collect()[0]

    sample_pdf = residual_df.limit(5000).toPandas()

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].hist(sample_pdf["residual"], bins=40, color="#00e5ff", edgecolor="#333355", alpha=0.85)
    axes[0].set_title(f"{model_name} - Residual Histogram")
    axes[0].set_xlabel("Residual")

    axes[1].scatter(sample_pdf["prediction"], sample_pdf["residual"], s=10, alpha=0.35, color="#ffab40")
    axes[1].axhline(0, color="#ff4081", linestyle="--", linewidth=1.2)
    axes[1].set_title(f"{model_name} - Residual vs Prediction")
    axes[1].set_xlabel("Prediction")
    axes[1].set_ylabel("Residual")

    plt.tight_layout()
    residual_plot_path = f"{PLOT_DIR}/residual_{model_name}.png"
    plt.savefig(residual_plot_path, bbox_inches="tight")
    plt.close(fig)

    return {
        "residual_mean": float(stats_row["residual_mean"] or 0.0),
        "residual_std": float(stats_row["residual_std"] or 0.0),
        "residual_abs_mean": float(stats_row["residual_abs_mean"] or 0.0),
    }, residual_plot_path


def actual_vs_pred_plot(pred_df, model_name):
    pdf = pred_df.select("popularity", "prediction").limit(5000).toPandas()

    fig, ax = plt.subplots(figsize=(7, 6))
    ax.scatter(pdf["popularity"], pdf["prediction"], s=10, alpha=0.35, color="#7c4dff")
    lims = [0, 100]
    ax.plot(lims, lims, "--", color="#ff4081", linewidth=1.2, label="Perfect")
    ax.set_title(f"{model_name} - Actual vs Predicted")
    ax.set_xlabel("Actual Popularity")
    ax.set_ylabel("Predicted Popularity")
    ax.legend()
    plt.tight_layout()

    path = f"{PLOT_DIR}/actual_vs_pred_{model_name}.png"
    plt.savefig(path, bbox_inches="tight")
    plt.close(fig)
    return path


def extract_feature_importance(fitted_model):
    if hasattr(fitted_model, "featureImportances"):
        vals = [float(x) for x in fitted_model.featureImportances.toArray()]
        method = "tree_feature_importance"
    elif hasattr(fitted_model, "coefficients"):
        vals = [abs(float(x)) for x in fitted_model.coefficients.toArray()]
        method = "abs_coefficients"
    else:
        return None, None

    if len(vals) != len(feature_cols):
        cols = [f"feature_{i}" for i in range(len(vals))]
    else:
        cols = feature_cols

    fi_df = pd.DataFrame({"feature": cols, "importance": vals}).sort_values("importance", ascending=False)
    return fi_df, method


def estimator_params(estimator):
    p = {}
    for param, value in estimator.extractParamMap().items():
        if isinstance(value, (str, int, float, bool)):
            p[param.name] = value
    return p


def log_feature_importance(run_name, fitted_model, human_name):
    fi_df, fi_method = extract_feature_importance(fitted_model)
    if fi_df is None:
        return None

    mlflow.log_param("feature_importance_method", fi_method)
    fi_csv = f"{PLOT_DIR}/feature_importance_{run_name}.csv"
    fi_df.to_csv(fi_csv, index=False)
    mlflow.log_artifact(fi_csv, artifact_path="feature_importance")

    top_n = fi_df.head(15)
    fig, ax = plt.subplots(figsize=(9, 6))
    ax.barh(top_n["feature"][::-1], top_n["importance"][::-1], color="#69f0ae", edgecolor="#333355")
    ax.set_title(f"{human_name} - Top 15 Feature Importance")
    ax.set_xlabel("Importance")
    plt.tight_layout()
    fi_plot = f"{PLOT_DIR}/feature_importance_{run_name}.png"
    plt.savefig(fi_plot, bbox_inches="tight")
    plt.close(fig)
    mlflow.log_artifact(fi_plot, artifact_path="feature_importance")

    model_importances[run_name] = fi_df
    return fi_df


## 6. Model 1: Linear Regression


In [12]:
lr = LinearRegression(
    featuresCol="features",
    labelCol="popularity",
    predictionCol="prediction",
    maxIter=100,
    regParam=0.1,
    elasticNetParam=0.0
)

with mlflow.start_run(run_name="Linear_Regression"):
    mlflow.log_param("model", "LinearRegression")
    mlflow.log_param("target", "popularity")
    mlflow.log_param("train_size", int(train_count))
    mlflow.log_param("test_size", int(test_count))
    mlflow.log_param("features_path", FEATURES_PATH)

    for k, v in estimator_params(lr).items():
        mlflow.log_param(k, v)

    lr_model = lr.fit(train_df)
    lr_preds = lr_model.transform(test_df)
    lr_metrics = evaluate_metrics(lr_preds)
    mlflow.log_metrics(lr_metrics)

    lr_residual_metrics, lr_res_fig = residual_stats_and_plot(lr_preds, "Linear Regression")
    for k, v in lr_residual_metrics.items():
        mlflow.log_metric(k, v)

    lr_avp_fig = actual_vs_pred_plot(lr_preds, "Linear Regression")
    mlflow.log_artifact(lr_res_fig, artifact_path="residual_analysis")
    mlflow.log_artifact(lr_avp_fig, artifact_path="prediction_plots")

    log_feature_importance("Linear_Regression", lr_model, "Linear Regression")
    mlflow.spark.log_model(lr_model, "linear_regression_model")

    lr_result = {
        "Model": "Linear_Regression",
        "rmse": round(lr_metrics["rmse"], 4),
        "mae": round(lr_metrics["mae"], 4),
        "r2": round(lr_metrics["r2"], 4),
        "residual_abs_mean": round(lr_residual_metrics["residual_abs_mean"], 4),
    }
    results.append(lr_result)

    print(f"Linear Regression -> RMSE: {lr_metrics['rmse']:.4f} | MAE: {lr_metrics['mae']:.4f} | R2: {lr_metrics['r2']:.4f}")


/opt/conda/lib/python3.11/site-packages/_distutils_hack/__init__.py:33: UserWarning: Setuptools is replacing distutils.
  warnings.warn("Setuptools is replacing distutils.")


Linear Regression -> RMSE: 21.2156 | MAE: 17.6092 | R2: 0.0615


## 7. Model 2: Decision Tree Regressor


In [ ]:
dt = DecisionTreeRegressor(
    featuresCol="features",
    labelCol="popularity",
    predictionCol="prediction",
    maxDepth=8,
    minInstancesPerNode=10
)

with mlflow.start_run(run_name="Decision_Tree_Regressor"):
    mlflow.log_param("model", "DecisionTreeRegressor")
    mlflow.log_param("target", "popularity")
    mlflow.log_param("train_size", int(train_count))
    mlflow.log_param("test_size", int(test_count))
    mlflow.log_param("features_path", FEATURES_PATH)

    for k, v in estimator_params(dt).items():
        mlflow.log_param(k, v)

    dt_model = dt.fit(train_df)
    dt_preds = dt_model.transform(test_df)
    dt_metrics = evaluate_metrics(dt_preds)
    mlflow.log_metrics(dt_metrics)

    dt_residual_metrics, dt_res_fig = residual_stats_and_plot(dt_preds, "Decision Tree")
    for k, v in dt_residual_metrics.items():
        mlflow.log_metric(k, v)

    dt_avp_fig = actual_vs_pred_plot(dt_preds, "Decision Tree")
    mlflow.log_artifact(dt_res_fig, artifact_path="residual_analysis")
    mlflow.log_artifact(dt_avp_fig, artifact_path="prediction_plots")

    dt_fi = log_feature_importance("Decision_Tree_Regressor", dt_model, "Decision Tree")
    mlflow.spark.log_model(dt_model, "decision_tree_model")

    dt_result = {
        "Model": "Decision_Tree_Regressor",
        "rmse": round(dt_metrics["rmse"], 4),
        "mae": round(dt_metrics["mae"], 4),
        "r2": round(dt_metrics["r2"], 4),
        "residual_abs_mean": round(dt_residual_metrics["residual_abs_mean"], 4),
    }
    results.append(dt_result)

    print(f"Decision Tree -> RMSE: {dt_metrics['rmse']:.4f} | MAE: {dt_metrics['mae']:.4f} | R2: {dt_metrics['r2']:.4f}")
    if dt_fi is not None:
        print("\nTop 10 Feature Importance (Decision Tree):")
        print(dt_fi.head(10).to_string(index=False))


## 8. Model 3: Random Forest Regressor


In [ ]:
rf = RandomForestRegressor(
    featuresCol="features",
    labelCol="popularity",
    predictionCol="prediction",
    numTrees=100,
    maxDepth=8,
    seed=42
)

with mlflow.start_run(run_name="Random_Forest_Regressor"):
    mlflow.log_param("model", "RandomForestRegressor")
    mlflow.log_param("target", "popularity")
    mlflow.log_param("train_size", int(train_count))
    mlflow.log_param("test_size", int(test_count))
    mlflow.log_param("features_path", FEATURES_PATH)

    for k, v in estimator_params(rf).items():
        mlflow.log_param(k, v)

    rf_model = rf.fit(train_df)
    rf_preds = rf_model.transform(test_df)
    rf_metrics = evaluate_metrics(rf_preds)
    mlflow.log_metrics(rf_metrics)

    rf_residual_metrics, rf_res_fig = residual_stats_and_plot(rf_preds, "Random Forest")
    for k, v in rf_residual_metrics.items():
        mlflow.log_metric(k, v)

    rf_avp_fig = actual_vs_pred_plot(rf_preds, "Random Forest")
    mlflow.log_artifact(rf_res_fig, artifact_path="residual_analysis")
    mlflow.log_artifact(rf_avp_fig, artifact_path="prediction_plots")

    rf_fi = log_feature_importance("Random_Forest_Regressor", rf_model, "Random Forest")
    mlflow.spark.log_model(rf_model, "random_forest_model")

    rf_result = {
        "Model": "Random_Forest_Regressor",
        "rmse": round(rf_metrics["rmse"], 4),
        "mae": round(rf_metrics["mae"], 4),
        "r2": round(rf_metrics["r2"], 4),
        "residual_abs_mean": round(rf_residual_metrics["residual_abs_mean"], 4),
    }
    results.append(rf_result)

    print(f"Random Forest -> RMSE: {rf_metrics['rmse']:.4f} | MAE: {rf_metrics['mae']:.4f} | R2: {rf_metrics['r2']:.4f}")
    if rf_fi is not None:
        print("\nTop 10 Feature Importance (Random Forest):")
        print(rf_fi.head(10).to_string(index=False))


## 9. Model 4: GBT Regressor


In [ ]:
gbt = GBTRegressor(
    featuresCol="features",
    labelCol="popularity",
    predictionCol="prediction",
    maxIter=100,
    maxDepth=6,
    stepSize=0.1,
    seed=42
)

with mlflow.start_run(run_name="GBT_Regressor"):
    mlflow.log_param("model", "GBTRegressor")
    mlflow.log_param("target", "popularity")
    mlflow.log_param("train_size", int(train_count))
    mlflow.log_param("test_size", int(test_count))
    mlflow.log_param("features_path", FEATURES_PATH)

    for k, v in estimator_params(gbt).items():
        mlflow.log_param(k, v)

    gbt_model = gbt.fit(train_df)
    gbt_preds = gbt_model.transform(test_df)
    gbt_metrics = evaluate_metrics(gbt_preds)
    mlflow.log_metrics(gbt_metrics)

    gbt_residual_metrics, gbt_res_fig = residual_stats_and_plot(gbt_preds, "GBT Regressor")
    for k, v in gbt_residual_metrics.items():
        mlflow.log_metric(k, v)

    gbt_avp_fig = actual_vs_pred_plot(gbt_preds, "GBT Regressor")
    mlflow.log_artifact(gbt_res_fig, artifact_path="residual_analysis")
    mlflow.log_artifact(gbt_avp_fig, artifact_path="prediction_plots")

    gbt_fi = log_feature_importance("GBT_Regressor", gbt_model, "GBT Regressor")
    mlflow.spark.log_model(gbt_model, "gbt_model")

    gbt_result = {
        "Model": "GBT_Regressor",
        "rmse": round(gbt_metrics["rmse"], 4),
        "mae": round(gbt_metrics["mae"], 4),
        "r2": round(gbt_metrics["r2"], 4),
        "residual_abs_mean": round(gbt_residual_metrics["residual_abs_mean"], 4),
    }
    results.append(gbt_result)

    print(f"GBT Regressor -> RMSE: {gbt_metrics['rmse']:.4f} | MAE: {gbt_metrics['mae']:.4f} | R2: {gbt_metrics['r2']:.4f}")
    if gbt_fi is not None:
        print("\nTop 10 Feature Importance (GBT):")
        print(gbt_fi.head(10).to_string(index=False))


## 10. Model 5: Generalized Linear Regression (GLR)


In [ ]:
glr = GeneralizedLinearRegression(
    featuresCol="features",
    labelCol="popularity",
    predictionCol="prediction",
    family="gaussian",
    link="identity",
    maxIter=100,
    regParam=0.1
)

with mlflow.start_run(run_name="Generalized_Linear_Regression"):
    mlflow.log_param("model", "GeneralizedLinearRegression")
    mlflow.log_param("target", "popularity")
    mlflow.log_param("train_size", int(train_count))
    mlflow.log_param("test_size", int(test_count))
    mlflow.log_param("features_path", FEATURES_PATH)

    for k, v in estimator_params(glr).items():
        mlflow.log_param(k, v)

    glr_model = glr.fit(train_df)
    glr_preds = glr_model.transform(test_df)
    glr_metrics = evaluate_metrics(glr_preds)
    mlflow.log_metrics(glr_metrics)

    glr_residual_metrics, glr_res_fig = residual_stats_and_plot(glr_preds, "GLR")
    for k, v in glr_residual_metrics.items():
        mlflow.log_metric(k, v)

    glr_avp_fig = actual_vs_pred_plot(glr_preds, "GLR")
    mlflow.log_artifact(glr_res_fig, artifact_path="residual_analysis")
    mlflow.log_artifact(glr_avp_fig, artifact_path="prediction_plots")

    glr_fi = log_feature_importance("Generalized_Linear_Regression", glr_model, "GLR")
    mlflow.spark.log_model(glr_model, "glr_model")

    glr_result = {
        "Model": "Generalized_Linear_Regression",
        "rmse": round(glr_metrics["rmse"], 4),
        "mae": round(glr_metrics["mae"], 4),
        "r2": round(glr_metrics["r2"], 4),
        "residual_abs_mean": round(glr_residual_metrics["residual_abs_mean"], 4),
    }
    results.append(glr_result)

    print(f"GLR -> RMSE: {glr_metrics['rmse']:.4f} | MAE: {glr_metrics['mae']:.4f} | R2: {glr_metrics['r2']:.4f}")
    if glr_fi is not None:
        print("\nTop 10 Feature Importance (GLR, abs coefficients):")
        print(glr_fi.head(10).to_string(index=False))


## 11. Model Karşılaştırma Tablosu


In [ ]:
results_df = pd.DataFrame(results).sort_values("rmse", ascending=True).reset_index(drop=True)
results_df


## 12. Performans Karşılaştırma Grafiği


In [ ]:
x = np.arange(len(results_df))
width = 0.25

fig, ax = plt.subplots(figsize=(13, 6))
bar1 = ax.bar(x - width, results_df["rmse"], width, label="RMSE", color=PALETTE[0], edgecolor="#333355")
bar2 = ax.bar(x,         results_df["mae"],  width, label="MAE",  color=PALETTE[1], edgecolor="#333355")
bar3 = ax.bar(x + width, results_df["r2"],   width, label="R2",   color=PALETTE[2], edgecolor="#333355")

ax.set_xticks(x)
ax.set_xticklabels(results_df["Model"], rotation=15, ha="right")
ax.set_title("Model Performance Comparison")
ax.set_ylabel("Metric Value")
ax.grid(axis="y", alpha=0.3)
ax.legend()

for bars in [bar1, bar2, bar3]:
    for b in bars:
        h = b.get_height()
        ax.text(b.get_x() + b.get_width() / 2, h + 0.15, f"{h:.2f}", ha="center", va="bottom", fontsize=7)

plt.tight_layout()
comparison_plot = f"{PLOT_DIR}/model_comparison.png"
plt.savefig(comparison_plot, bbox_inches="tight")
plt.show()


## 13. En İyi Modelin Feature Importance Grafiği


In [ ]:
best_model = results_df.iloc[0]["Model"]
print(f"En iyi model (RMSE): {best_model}")

if best_model in model_importances:
    best_fi = model_importances[best_model].head(15)

    fig, ax = plt.subplots(figsize=(10, 7))
    colors = plt.cm.cool(np.linspace(0.2, 0.9, len(best_fi)))
    ax.barh(best_fi["feature"][::-1], best_fi["importance"][::-1], color=colors, edgecolor="#333355")
    ax.set_xlabel("Importance Score")
    ax.set_title(f"Feature Importance - {best_model}")
    ax.grid(axis="x", alpha=0.3)
    plt.tight_layout()

    best_fi_plot = f"{PLOT_DIR}/best_model_feature_importance.png"
    plt.savefig(best_fi_plot, bbox_inches="tight")
    plt.show()
else:
    best_fi_plot = None
    print("Bu model için feature importance bulunamadı.")


## 14. MLflow'a Özet Run Loglama


In [ ]:
results_csv = f"{REPORT_DIR}/step6_regression_results.csv"
results_df.to_csv(results_csv, index=False)

with mlflow.start_run(run_name="Model_Comparison_Summary"):
    for _, row in results_df.iterrows():
        name = row["Model"].replace(" ", "_")
        mlflow.log_metric(f"{name}_rmse", float(row["rmse"]))
        mlflow.log_metric(f"{name}_mae", float(row["mae"]))
        mlflow.log_metric(f"{name}_r2", float(row["r2"]))

    mlflow.log_param("best_model", best_model)
    mlflow.log_param("target", "popularity")
    mlflow.log_param("features_path", FEATURES_PATH)

    mlflow.log_artifact(results_csv, artifact_path="summary")
    mlflow.log_artifact(comparison_plot, artifact_path="summary")
    if best_fi_plot is not None:
        mlflow.log_artifact(best_fi_plot, artifact_path="summary")

print(f"Sonuç CSV: {results_csv}")
print(f"MLflow UI: {MLFLOW_URI}")


## 15. Sonuç

Kontrol etmen gerekenler:
1. MLflow'da 5 model run'ı oluştu mu?
2. Her run'da RMSE / MAE / R2 metrikleri var mı?
3. Her run'da model artifact'i var mı?
4. Residual grafik ve feature importance artifact'leri oluştu mu?


In [ ]:
print("Pipeline tamamlandı.")
print("Öneri: MLflow arayüzünde modelleri RMSE'ye göre karşılaştır.")
